## Ingest pit_stops.json


### Step 0 - Parameters and Imports


In [0]:
%run "../utils/config"


In [0]:
dbutils.widgets.text("p_source_value", "")
v_source_value = dbutils.widgets.get("p_source_value")


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType, TimestampType, FloatType
from pyspark.sql.functions import col, lit, current_timestamp, to_timestamp, concat

### Step 1 - Read the json file


#### 1.1 Define the schema


In [0]:
pitstops_schema = StructType([
    StructField("driverId", StringType(), False),
    StructField("duration", FloatType(), True),
    StructField("lap", IntegerType(), True),
    StructField("milliseconds", IntegerType(), True),
    StructField("raceId", IntegerType(), True),
    StructField("stop", IntegerType(), True),
    StructField("time", StringType(), True)
])


#### 1.2 Read the json file


In [ ]:
pitstops_df = (spark.read
               .format("json")
               .schema(pitstops_schema)
               .option("multiLine", True)
               .load(f"{raw_folder_path}/pit_stops.json")
               )


### Step 2 - Transform the data


In [0]:
renamed_pistops_df = add_ingestion_date(
    pitstops_df
        .withColumnRenamed("raceId", "race_id")
        .withColumnRenamed("driverId", "driver_id")
        .withColumn("source", lit(v_source_value)))


### Step 3 - Write the output to parquet


In [ ]:
renamed_pistops_df.write.format("delta").mode("overwrite").saveAsTable("f1_processed.pit_stops")


In [0]:
dbutils.notebook.exit("success")